In [10]:
import os
import numpy as np
import scipy.stats as stats
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from dynamic_routing_analysis import decoding_utils
import upath
# import dynamic_routing_analysis as dra
# import npc_lims

import matplotlib
import matplotlib.font_manager as fm

matplotlib.rcParams['font.size'] = 8
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
fm.FontProperties().set_family('arial')

%load_ext autoreload
%autoreload 2
%matplotlib inline
# %matplotlib widget

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
session_table_path=r"\\allen\programs\mindscope\workgroups\dynamicrouting\Ethan\CO decoding results\session_table_v0.289.parquet"
session_table=pl.read_parquet(session_table_path)

dr_session_list=(
    session_table.filter(
    pl.col('project')=="DynamicRouting",
    pl.col('is_production'),
    pl.col('is_annotated'),
    pl.col('is_ephys'),
    pl.col('is_opto_perturbation').eq(False),
    pl.col('is_injection_perturbation').eq(False),
    pl.col('is_context_naive').eq(False),
    pl.col('issues')==[],
    # pl.col('is_good_behavior').eq(True),
    # pl.col('is_engaged').eq(True),
    )['session_id'].to_list()
    )

block_dprime_threshold=1.0

good_behavior_table={
    'session_id':[],
    'n_good_vis_blocks':[],
    'n_good_aud_blocks':[],
    'n_engaged_blocks':[],
}

for sel_session in dr_session_list:
    n_good_vis_blocks=np.sum(session_table.filter(pl.col('session_id') == sel_session)['cross_modality_dprime_vis_blocks'].to_numpy()[0]>=block_dprime_threshold)
    n_good_aud_blocks=np.sum(session_table.filter(pl.col('session_id') == sel_session)['cross_modality_dprime_aud_blocks'].to_numpy()[0]>=block_dprime_threshold)
    good_behavior_table['session_id'].append(sel_session)
    good_behavior_table['n_good_vis_blocks'].append(n_good_vis_blocks)
    good_behavior_table['n_good_aud_blocks'].append(n_good_aud_blocks)
    n_engaged_blocks=np.sum(session_table.filter(pl.col('session_id') == sel_session)['n_contingent_rewards'].to_numpy()[0]>=10)
    good_behavior_table['n_engaged_blocks'].append(n_engaged_blocks)

good_behavior_table=pd.DataFrame(good_behavior_table)
dr_session_list=good_behavior_table.query('n_good_vis_blocks>=2 and n_good_aud_blocks>=2 and n_engaged_blocks>=4')['session_id'].values

In [3]:
residuals_path="s3://aind-scratch-data/dynamic-routing/glm_residuals/"

In [4]:
# residuals=pl.scan_parquet(residuals_path,storage_options={"skip_signature": "true"})
# residuals.columns

In [5]:
sel_session='626791_2022-08-15'
test=pd.read_parquet(os.path.join(residuals_path,"GLM_residuals_facialfeatures_" + sel_session + ".parquet"))

In [6]:
test

,trial_no,626791_2022-08-15_A-511,626791_2022-08-15_C-188,626791_2022-08-15_C-193,626791_2022-08-15_A-334,626791_2022-08-15_A-292,626791_2022-08-15_F-362,626791_2022-08-15_B-136,626791_2022-08-15_F-701,626791_2022-08-15_C-153,...,626791_2022-08-15_A-240,626791_2022-08-15_C-228,626791_2022-08-15_A-248,626791_2022-08-15_F-122,626791_2022-08-15_F-104,626791_2022-08-15_F-781,626791_2022-08-15_B-298,626791_2022-08-15_B-21,626791_2022-08-15_F-825,context
0,0,-0.465759,-0.866052,0.970140,-1.075352,-0.206620,-0.074364,-1.629696,-0.447628,1.248618,...,-0.611805,-0.360186,-0.976714,-0.362434,-0.962642,-0.231085,-0.432894,-0.440909,0.876348,vis
1,1,-0.213664,-0.975409,-1.156161,-0.605836,0.281973,0.092111,-0.780103,-0.723099,-0.814669,...,-0.002054,-0.397605,-0.631073,0.761354,-0.733360,-0.139084,-0.376782,-0.368314,1.720880,vis
2,2,0.810546,-0.575216,-0.412828,-1.058470,0.084538,0.484327,-0.467614,-1.006985,3.367816,...,-0.301795,-0.766444,0.022571,-0.119624,0.100525,0.180861,-1.110615,0.020533,1.248698,vis
3,3,-0.804410,-1.596928,-0.020192,-0.825785,-0.114203,-0.678747,0.775437,-1.553000,1.367697,...,-0.539155,-0.344192,0.195655,-0.051980,0.157656,0.364187,-1.055784,-0.685085,0.186042,vis
4,4,0.271278,-0.411154,-0.546759,-1.023811,-0.792633,0.901068,-1.178791,-1.359220,1.397070,...,0.200678,-0.898917,-0.184028,-0.188241,-0.802419,0.026683,-0.947632,-0.319285,1.974685,vis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
474,474,-1.018503,-0.698387,0.660557,-1.121186,-0.410259,0.006880,0.793030,-0.603651,0.453269,...,-0.273328,1.630514,0.062162,-0.392383,-0.278900,-0.296022,-0.133665,0.053797,-0.593714,aud
475,475,-0.601337,-0.242885,1.067427,-0.268629,-0.479734,0.038498,0.154413,0.157504,-0.361561,...,-0.119975,-0.009309,1.144694,-0.453006,-0.374620,1.902063,0.059357,2.321872,0.094972,aud
476,476,2.507598,-0.517361,2.454154,-0.411726,-0.736382,0.428072,-0.340155,-0.173111,0.147965,...,-0.423982,1.701506,0.249575,-0.409392,-1.414064,-0.201724,-0.232217,0.047494,-0.513548,aud
477,477,-0.929119,0.639903,-0.618473,0.864067,-0.148616,0.114369,0.669812,-0.286741,-0.088354,...,-1.014430,-0.276035,0.114632,-0.461427,-0.552434,-0.276886,-0.163336,-0.388363,-0.607345,aud


In [30]:
# input_df=test
# input_df.rename(columns={'trial_no':'trial_index'},inplace=True)
# input_df.drop(columns=['context'],inplace=True)
# spike_counts_df=input_df.unpivot(
#     index='trial_index',
#     columns=[col for col in input_df.columns if col != 'trial_index'],
#     variable_name='unit_id',
#     value_name='n_spikes_window'
# )

In [7]:
input_df=test.drop(columns=['context']).rename(columns={'trial_no':'trial_index'})

spike_counts_df=pl.from_pandas(input_df.melt(
    id_vars='trial_index',
    value_vars=[col for col in input_df.columns if col != 'trial_index'],
    var_name='unit_id',
    value_name='n_spikes_window'
).sort_values(by=['trial_index','unit_id']).reset_index(drop=True))

In [8]:
spike_counts_df

trial_index,unit_id,n_spikes_window
i64,str,f64
0,"""626791_2022-08-15_A-103""",2.093221
0,"""626791_2022-08-15_A-106""",0.503682
0,"""626791_2022-08-15_A-107""",-0.891468
0,"""626791_2022-08-15_A-108""",-0.896718
0,"""626791_2022-08-15_A-109""",1.241157
…,…,…
478,"""626791_2022-08-15_F-905""",-0.44425
478,"""626791_2022-08-15_F-916""",-0.324601
478,"""626791_2022-08-15_F-946""",-0.079326


In [ ]:
# import upath

In [ ]:
# save_path.mkdir(parents=True, exist_ok=True)

In [17]:
save_path=upath.UPath("s3://aind-scratch-data/dynamic-routing/decoding/GLM_residuals_reformatted_v289")

# save_path=upath.UPath(r"\\allen\programs\mindscope\workgroups\dynamicrouting\Ethan\GLM_residuals_reformatted")

residuals_path=upath.UPath("s3://aind-scratch-data/dynamic-routing/glm_residuals/")

for session in dr_session_list[:]:
    print(f"Processing session {session}")
    try:
        # input_df=pd.read_parquet(os.path.join(residuals_path,"GLM_residuals_facialfeatures_" + session + ".parquet"))
        input_df=pd.read_parquet(os.path.join(residuals_path,f"{session}_v289_movementfeatures_1.parquet"))
        input_df=input_df.drop(columns=['context']).rename(columns={'trial_no':'trial_index'})
        spike_counts_df=pl.from_pandas(input_df.melt(
            id_vars='trial_index',
            value_vars=[col for col in input_df.columns if col != 'trial_index'],
            var_name='unit_id',
            value_name='n_spikes_window'
        ).sort_values(by=['trial_index','unit_id']).reset_index(drop=True))
        
        save_file=save_path / ("GLM_residuals_reformatted_" + session + ".parquet")
        print(f"Saving to {save_file}")
        spike_counts_df.write_parquet(save_file)
    except Exception as e:
        print(f"Failed to process session {session}: {e}")

Processing session 626791_2022-08-15
Saving to s3://aind-scratch-data/dynamic-routing/decoding/GLM_residuals_reformatted_v289/GLM_residuals_reformatted_626791_2022-08-15.parquet
Processing session 644864_2023-01-30
Saving to s3://aind-scratch-data/dynamic-routing/decoding/GLM_residuals_reformatted_v289/GLM_residuals_reformatted_644864_2023-01-30.parquet
Processing session 644864_2023-01-31
Saving to s3://aind-scratch-data/dynamic-routing/decoding/GLM_residuals_reformatted_v289/GLM_residuals_reformatted_644864_2023-01-31.parquet
Processing session 644864_2023-02-02
Saving to s3://aind-scratch-data/dynamic-routing/decoding/GLM_residuals_reformatted_v289/GLM_residuals_reformatted_644864_2023-02-02.parquet
Processing session 644866_2023-02-07
Saving to s3://aind-scratch-data/dynamic-routing/decoding/GLM_residuals_reformatted_v289/GLM_residuals_reformatted_644866_2023-02-07.parquet
Processing session 644866_2023-02-08
Saving to s3://aind-scratch-data/dynamic-routing/decoding/GLM_residuals_r

In [12]:
# save_file

In [18]:
# sel_session="664851_2023-11-16"
# test=pl.read_parquet(r"\\allen\programs\mindscope\workgroups\dynamicrouting\Ethan\GLM_residuals_reformatted\GLM_residuals_reformatted_664851_2023-11-16.parquet")
test=pl.read_parquet(r"s3://aind-scratch-data/dynamic-routing/decoding/GLM_residuals_reformatted_v289/GLM_residuals_reformatted_626791_2022-08-15.parquet")
test

trial_index,unit_id,n_spikes_window
i64,str,f64
0,"""626791_2022-08-15_A-103""",1.039036
0,"""626791_2022-08-15_A-106""",0.509699
0,"""626791_2022-08-15_A-107""",-1.183988
0,"""626791_2022-08-15_A-113""",-0.049349
0,"""626791_2022-08-15_A-114""",0.485067
…,…,…
478,"""626791_2022-08-15_F-905""",-1.125422
478,"""626791_2022-08-15_F-916""",-0.642646
478,"""626791_2022-08-15_F-946""",0.063663


In [ ]:
sel_session='664851_2023-11-16'
glm_residuals_path="\\\\allen\\programs\\mindscope\\workgroups\\dynamicrouting\\Ethan\\GLM_residuals_reformatted\\"
glm_residuals=pl.scan_parquet(glm_residuals_path)